# 03 — OCR Artifact Validation

**Purpose:** Decide whether OCR artifact filtering is necessary before quantitative text analysis.

This notebook runs on the output of `02_xml_parsing.ipynb` and checks:
1. Basic noise profile (character-level)
2. Common OCR artifact patterns
3. Short/suspicious region candidates
4. Token-level sanity check (word length distribution)
5. A verdict summary

**You do not need to filter** if the artifact rate is low and patterns are minor.

## Setup — Re-run parsing from notebook 02
Paste your path and re-use the same functions.

In [1]:
from lxml import etree
from pathlib import Path
import re
import pandas as pd
import collections

# ── adjust this path ──
data_dir = Path("/Users/sophiehamann/Documents/MA_Universität_Wien/master_thesis/master_thesis_data/xml_data")

ns = {"page": "http://schema.primaresearch.org/PAGE/gts/pagecontent/2013-07-15"}
issue_folders = sorted(data_dir.glob("heresies_*_pagexml"))
print(f"Found {len(issue_folders)} issue folder(s)")

Found 11 issue folder(s)


In [2]:
# Copy of parse helpers from notebook 02

def parse_custom(custom_string):
    type_match = re.search(r"type:([^;}\s]+)", custom_string)
    structure_type = type_match.group(1) if type_match else None
    order_match = re.search(r"readingOrder\s*\{index:(\d+);?\}", custom_string)
    reading_order = int(order_match.group(1)) if order_match else None
    return {"type": structure_type, "reading_order": reading_order}


def extract_regions_from_page(page_file, ns, issue_num, page_num):
    tree = etree.parse(str(page_file))
    root = tree.getroot()
    raw_regions = []
    for region in root.findall(".//page:TextRegion", ns):
        custom_string = region.get("custom", "")
        parsed = parse_custom(custom_string)
        lines = []
        for line in region.findall(".//page:TextLine", ns):
            unicode_el = line.find(".//page:Unicode", ns)
            if unicode_el is not None and unicode_el.text:
                lines.append(unicode_el.text)
        raw_regions.append({
            "issue": issue_num,
            "page": page_num,
            "id": region.get("id"),
            "type": parsed["type"],
            "reading_order": parsed["reading_order"],
            "lines": lines,
            "raw_text": " ".join(lines)
        })
    raw_regions.sort(key=lambda r: (r["reading_order"] is None, r["reading_order"]))
    return raw_regions


# Load all regions from all issues
all_regions = []
for folder in issue_folders:
    issue_num = re.search(r"heresies_(\d+)_pagexml", folder.name).group(1)
    page_files = sorted(folder.glob("page_*.xml"),
                        key=lambda f: int(re.search(r"page_(\d+)", f.stem).group(1)))
    for page_file in page_files:
        page_num = int(re.search(r"page_(\d+)", page_file.stem).group(1))
        all_regions.extend(extract_regions_from_page(page_file, ns, issue_num, page_num))

print(f"Total regions loaded: {len(all_regions)}")

Total regions loaded: 13721


## 1 — Noise Profile: Character-level

We count non-standard characters — anything outside plain ASCII letters, digits, and common punctuation. A high rate signals OCR noise.

In [3]:
# Characters we consider 'normal' for an English-language art magazine
NORMAL_CHARS = re.compile(r"[a-zA-Z0-9\s.,;:!?'\"()\-–—/&%$@#]") 

all_text = " ".join(r["raw_text"] for r in all_regions)
total_chars = len(all_text)
noise_chars = [c for c in all_text if not NORMAL_CHARS.match(c)]
noise_counter = collections.Counter(noise_chars)

print(f"Total characters:  {total_chars:,}")
print(f"Noise characters:  {len(noise_chars):,}")
print(f"Noise rate:        {len(noise_chars)/total_chars:.2%}")
print()
print("Top 20 noise characters:")
for char, count in noise_counter.most_common(20):
    print(f"  {repr(char):12s}  {count:>6,}")

Total characters:  5,331,987
Noise characters:  2,567
Noise rate:        0.05%

Top 20 noise characters:
  '´'            1,450
  ']'              336
  '['              324
  'é'              232
  'ñ'               60
  '’'               46
  '¬'               21
  'á'               18
  '*'               15
  'à'               10
  '¼'               10
  '§'               10
  '“'                9
  'ö'                9
  '+'                3
  '•'                3
  'ï'                2
  '‚'                2
  'è'                2
  'ä'                2


**Interpret:** A noise rate below ~1–2% is generally fine for frequency-based text analysis. Some accented characters (é, ü) or ligatures may appear and are often legitimate content from a 1970s art magazine (artist names, foreign words).

## 2 — Common OCR Artifact Patterns

We test for well-known Transkribus/OCR failure modes.

In [4]:
ARTIFACT_PATTERNS = {
    "Repeated single char (e.g. lllll)": re.compile(r"(.)\1{4,}"),
    "Long string without spaces (>25 chars)": re.compile(r"\S{25,}"),
    "Mixed digit+letter garbage (e.g. a3b7c)": re.compile(r"(?:[a-zA-Z]\d|\d[a-zA-Z]){3,}"),
    "Pipe or box chars (| ¦ │)": re.compile(r"[|¦│]"),
    "Ligature artifacts (ﬁ ﬂ ﬀ)": re.compile(r"[ﬁﬂﬀﬃﬄ]"),
    "Multiple consecutive hyphens": re.compile(r"-{3,}"),
    "Stray numbers alone on a line": re.compile(r"^\s*\d{1,3}\s*$", re.MULTILINE),
}

print(f"{'Pattern':<45} {'Regions hit':>12} {'Examples'}")
print("-" * 90)

for name, pattern in ARTIFACT_PATTERNS.items():
    hits = [(r, pattern.search(r["raw_text"])) for r in all_regions]
    hits = [(r, m) for r, m in hits if m]
    examples = "; ".join(m.group()[:30] for _, m in hits[:3])
    print(f"{name:<45} {len(hits):>12,}    {examples}")

Pattern                                        Regions hit Examples
------------------------------------------------------------------------------------------
Repeated single char (e.g. lllll)                       11    RRRRRRRRRR; .....; .....
Long string without spaces (>25 chars)                  32    socialist-feminist-artist; politicaleconomic/cultural; proletariat...recognize...that
Mixed digit+letter garbage (e.g. a3b7c)                  1    M3J2R2
Pipe or box chars (| ¦ │)                                0    
Ligature artifacts (ﬁ ﬂ ﬀ)                               0    
Multiple consecutive hyphens                             1    ----
Stray numbers alone on a line                        1,171    2; 3; 4


In [5]:
# Show actual examples for the most interesting patterns — edit as needed
TARGET = "Long string without spaces (>25 chars)"
pattern = ARTIFACT_PATTERNS[TARGET]

print(f"Up to 10 example regions matching: {TARGET}\n")
count = 0
for r in all_regions:
    m = pattern.search(r["raw_text"])
    if m:
        print(f"  Issue {r['issue']} p.{r['page']} | type: {r['type']}")
        print(f"  Text: {r['raw_text'][:200]}")
        print()
        count += 1
        if count >= 10:
            break

Up to 10 example regions matching: Long string without spaces (>25 chars)

  Issue 01 p.5 | type: paragraph
  Text: What kind of socialist-feminist-artist am 1? What kind of socialist artist loves Corot as well as Courbet and forgives oil painting its bourgeois origins and abstract expressionism its heraldry of U.S

  Issue 01 p.8 | type: paragraph
  Text: So, as Marxists, we come to feminism from a completely different place than the "mechani- cal Marxists." Because we see monopoly capi- talism as a politicaleconomic/cultural totality, we have room wit

  Issue 02 p.53 | type: paragraph
  Text: Tristan's major work, The Union of Labor, appeared in 1843, shortly before her death. Long before Marx she proposed and ex- pounded her idea that although the emancipation of workers must be achieved 

  Issue 02 p.103 | type: paragraph
  Text: Ardele Lister is a Canadian feminist/artist/filmmaker critic and the editor of Criteria, published in Vancouver. Last ycar she co-scripted and directed 

## 3 — Short / Suspicious Region Candidates

In [6]:
# Regions with very little text — likely captions, page numbers, or noise
short_regions = [r for r in all_regions if 0 < len(r["raw_text"].strip()) < 10]

print(f"Regions with 1–9 characters: {len(short_regions)} ({len(short_regions)/len(all_regions):.1%} of total)\n")

# Distribution by region type
type_counts = collections.Counter(r["type"] for r in short_regions)
print("By region type:")
for t, c in type_counts.most_common():
    print(f"  {str(t):<30} {c}")

print("\nSample short regions:")
for r in short_regions[:15]:
    print(f"  Issue {r['issue']} p.{r['page']} | {r['type']:<25} | {repr(r['raw_text'].strip())}")

Regions with 1–9 characters: 1308 (9.5% of total)

By region type:
  page_number                    1147
  heading                        77
  paragraph                      38
  caption                        21
  page-number                    15
  author_creator                 9
  None                           1

Sample short regions:
  Issue 01 p.4 | page_number               | '2'
  Issue 01 p.5 | page_number               | '3'
  Issue 01 p.6 | page_number               | '4'
  Issue 01 p.7 | page_number               | '5'
  Issue 01 p.8 | page_number               | '6'
  Issue 01 p.9 | page_number               | '7'
  Issue 01 p.10 | page_number               | '8'
  Issue 01 p.11 | page_number               | '9'
  Issue 01 p.12 | page_number               | '10'
  Issue 01 p.13 | page_number               | '11'
  Issue 01 p.14 | page_number               | '12'
  Issue 01 p.15 | page_number               | '13'
  Issue 01 p.16 | page_number               | '14'
  Issue 0

## 4 — Word Length Distribution

A healthy OCR output has mostly 2–12 character words. Very long 'words' often indicate run-together lines.

In [7]:
words = re.findall(r"\S+", all_text)
word_lengths = [len(w) for w in words]

length_counter = collections.Counter(word_lengths)
df_lengths = pd.DataFrame(
    [(l, c, c / len(words)) for l, c in sorted(length_counter.items())],
    columns=["length", "count", "share"]
)

print(f"Total tokens: {len(words):,}")
print(f"Mean word length: {sum(word_lengths)/len(word_lengths):.2f}")
print()
print("Word length distribution (tokens):")
print(df_lengths[df_lengths["length"] <= 30].to_string(index=False))

long_words = [w for w in words if len(w) > 20]
print(f"\nTokens longer than 20 chars: {len(long_words):,} ({len(long_words)/len(words):.2%})")
if long_words:
    print("Sample:", long_words[:10])

Total tokens: 910,407
Mean word length: 4.86

Word length distribution (tokens):
 length  count    share
      1  33080 0.036335
      2 145994 0.160361
      3 176228 0.193571
      4 136682 0.150133
      5 110014 0.120840
      6  81990 0.090059
      7  71599 0.078645
      8  54162 0.059492
      9  38233 0.041996
     10  26708 0.029336
     11  16227 0.017824
     12   9301 0.010216
     13   5411 0.005943
     14   2430 0.002669
     15   1116 0.001226
     16    524 0.000576
     17    287 0.000315
     18    137 0.000150
     19    102 0.000112
     20     45 0.000049
     21     44 0.000048
     22     29 0.000032
     23     16 0.000018
     24     16 0.000018
     25     12 0.000013
     26      2 0.000002
     27      1 0.000001
     28      2 0.000002
     30      1 0.000001

Tokens longer than 20 chars: 137 (0.02%)
Sample: ['socialist-feminist-artist', 'politicaleconomic/cultural', 'capitalist-controlled', 'neighborhood-oriented', 'esthetic/sociological', 'consciousness

## 5 — Verdict Summary

In [8]:
# Compute key metrics for a quick verdict
noise_rate = len(noise_chars) / total_chars
short_rate = len(short_regions) / len(all_regions)
long_word_rate = len(long_words) / len(words)

print("=" * 55)
print("OCR ARTIFACT VALIDATION — SUMMARY")
print("=" * 55)
print(f"  Noise character rate:   {noise_rate:.2%}")
print(f"  Short region rate:      {short_rate:.2%}")
print(f"  Long-token rate (>20c): {long_word_rate:.2%}")
print()

flags = []
if noise_rate > 0.02:
    flags.append("⚠ Noise rate above 2% — consider character normalization")
if long_word_rate > 0.01:
    flags.append("⚠ Many long tokens — possible line-merge errors or URLs")
if short_rate > 0.15:
    flags.append("⚠ Many very short regions — check if region type filtering is sufficient")

if flags:
    print("FLAGS:")
    for f in flags:
        print(" ", f)
else:
    print("✓ No major issues detected.")
    print("  Your existing region-type filtering is likely sufficient.")
    print("  OCR artifact filtering is probably NOT necessary.")
print()
print("Note: Always manually spot-check a few pages regardless of metrics.")

OCR ARTIFACT VALIDATION — SUMMARY
  Noise character rate:   0.05%
  Short region rate:      9.53%
  Long-token rate (>20c): 0.02%

✓ No major issues detected.
  Your existing region-type filtering is likely sufficient.
  OCR artifact filtering is probably NOT necessary.

Note: Always manually spot-check a few pages regardless of metrics.
